In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from classification_training_reporter import TrainingReporter
from classification_xbg_model import build_model_from_grid_params
from classification_dataset_preprocessing import make_preprocessing_pipeline, make_label_pipeline, make_training_pipeline, make_delta_pipeline, make_standarize_pipeline
from classification_xbg_model import CustomXGBClassifierModel


# Odczyt datasetu

In [2]:
df = pd.read_csv(os.path.join("../data", "ortodoncja.csv"))

# Podział danych na zbiór treningowy i testowy

In [3]:
pipeline = make_label_pipeline()
df_processed = pipeline.fit_transform(df)
label_encoder = pipeline.named_steps["encode_labels"].encoder

X = df_processed.drop(columns=["growth direction"])
y = df_processed["growth direction"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tpipeline = make_training_pipeline()
X_train, y_train = tpipeline.fit_resample(X_train,y_train)
X_test = tpipeline[:1].fit_transform(X_test)

# Inicjalizacja reportera do treningów

In [4]:
model = CustomXGBClassifierModel()
reporter = TrainingReporter(model, X_train, X_test, y_train, y_test)

# Pierwszy trening

In [5]:
reporter.train()

Start training...
Training finished!
Train Accuracy: 0.9949  |  Test Accuracy: 0.7667
Train F1:       0.9949  |  Test F1:       0.7667
Train AUROC:    0.9999  |  Test AUROC:    0.8679
---------------------------------------------------


# Cross walidacja

In [6]:
reporter.run_cross_validation(cv=10)

Start cross validation...
Fold 0:
  Train Accuracy: 0.9925  |  Val Accuracy: 0.8833
  Train F1:       0.9925  |  Val F1:       0.8815
---------------------------------------------------
Fold 1:
  Train Accuracy: 0.9906  |  Val Accuracy: 0.8833
  Train F1:       0.9906  |  Val F1:       0.8823
---------------------------------------------------
Fold 2:
  Train Accuracy: 0.9963  |  Val Accuracy: 0.8500
  Train F1:       0.9963  |  Val F1:       0.8476
---------------------------------------------------
Fold 3:
  Train Accuracy: 0.9963  |  Val Accuracy: 0.8667
  Train F1:       0.9963  |  Val F1:       0.8653
---------------------------------------------------
Fold 4:
  Train Accuracy: 1.0000  |  Val Accuracy: 0.8475
  Train F1:       1.0000  |  Val F1:       0.8454
---------------------------------------------------
Fold 5:
  Train Accuracy: 0.9963  |  Val Accuracy: 0.8983
  Train F1:       0.9963  |  Val F1:       0.8965
---------------------------------------------------
Fold 6:
  Trai

# Randomized grid search

In [ ]:
random_grid = reporter.run_randomized_search_xbg(cv=5)

Start randomized grid search for XBGClassifier...


# Kroswalidacja po dostosowaniu hiperparametrów za pomocą Randomized Grid Search

In [ ]:
model_RGS = build_model_from_grid_params(random_grid.best_params_)
reporter_RGS = TrainingReporter(model_RGS, X_train, X_test, y_train, y_test)
reporter_RGS.run_cross_validation(cv=10)

# Totalny overfitting